## OOPS Basics

- The `__init__` method is a special method used to initialize newly created objects. Often referred to as the constructor, it is automatically invoked when a new instance of a class is created. This method allows you to set up the initial state of the object by assigning values to its attributes.

- the `self` keyword is used within a class to refer to the current instance of that class. It allows methods to access and modify the attributes and methods associated with the specific object.

In [35]:
class Person:
    #class attributes - shared among all instances
    business_unit = "Shared Capabilities"
    skill_family = "Python"

    def __init__(self,name,age):
        #instance attributes
        self.name = name
        self.age = age
        self.business_unit = "Default Unit"

    #instance methods
    def describe(self): 
        # inbuilt method to return attributes as dict
        return vars(self)
    
    def greet(self):
        # cls attributes accessed via current instance of cls
        # when accessed via current instance, python first looks for inst attribute 
        # if not found will revert to cls attribute
        # Hence always use cls name to access cls attribute
        print(f"Hi, I am {self.name} from business_unit - {self.business_unit}")

        # cls attribute accessed via classname
        print(f"My primary skillset is - {Person.skill_family}")

#obj initialization - refers to current instance of cls
person1 = Person("Arun",35)
person2 = Person("Aditya",22)

In [25]:
person1.describe()

{'name': 'Arun', 'age': 35}

In [34]:
person1.greet()

Hi, I am Arun from business_unit - Default Unit
My primary skillset is - Python


In [7]:
person2.describe()

{'name': 'Aditya', 'age': 22}

In [16]:
person1.name = "Arjun"

In [30]:
Person.business_unit = "Advanced Technologies"

In [17]:
person1.describe()

{'name': 'Arjun', 'age': 35}

In [31]:
person1.greet()

Hi, I am Arun from business_unit - Advanced Technologies
My primary skillset is - Python


In [32]:
person2.greet()

Hi, I am Aditya from business_unit - Advanced Technologies
My primary skillset is - Python


# Useful Refs.:

- https://www.freecodecamp.org/news/solid-principles-explained-in-plain-english/


# ✅ S.O.L.I.D Principles – Deep Dive with Real-World Scenarios

### 🔹 **1. Single Responsibility Principle (SRP)**

> **"A class should have only one reason to change."**

#### ✅ Example: **Amazon - Order Module**
- `Order` class should not:
  - Calculate prices
  - Send confirmation emails
  - Update inventory
- Instead:
  - `Order` handles order state (placed, shipped)
  - `InvoiceService` handles price calculation
  - `EmailService` sends notifications

#### ❌ Violation Scenario:
```python
class Order:
    def place_order(self, cart, user):
        # logic to validate cart
        # logic to update inventory
        # logic to send email
```
➡️ This class changes if **email logic** changes, or **inventory rules** change.

#### ✅ Fix:
Break into:
- `OrderProcessor`
- `InventoryManager`
- `NotificationService`

#### 🔁 Common Gotcha:
- Trying to do too much in a "manager" or "service" class.
- Watch out for God Classes.

#### 🎯 Interview Questions:
- How do you identify if a class has multiple responsibilities?
    - Answer: If a class has more than one reason to change — for example, an Order class in an e-commerce system that also sends emails and manages inventory — it violates SRP. Changing business rules for email shouldn't touch order logic.

- Give a time when splitting a class helped with maintainability.
    - Answer: In a library system, splitting BookService into BookSearchService and BookInventoryService helped us decouple search logic from availability, which made changes easier when we integrated Elasticsearch.

- Have you ever dealt with a God class?
    - Answer: Yes, in a legacy movie booking system, the BookingManager handled search, seat allocation, payment, and ticket generation. We broke it into separate services — SearchService, PaymentGateway, TicketGenerator — to isolate concerns and test them independently.

---

### 🔹 **2. Open/Closed Principle (OCP)**

> **"Software entities should be open for extension but closed for modification."**

#### ✅ Example: **ATM - Transactions**
Suppose we support:
- `Withdraw`
- `Deposit`

Later, we want to add:
- `BalanceCheck`
- `MiniStatement`

#### ❌ Violation:
```python
class ATMTransaction:
    def process(self, type):
        if type == 'withdraw':
            # logic
        elif type == 'deposit':
            # logic
```
➡️ Modifying this every time = **fragile code**.

#### ✅ Fix:
Use **Inheritance + Polymorphism**:
```python
class Transaction:
    def process(self): pass

class Withdraw(Transaction): ...
class Deposit(Transaction): ...
class BalanceCheck(Transaction): ...
```

#### 🔁 Common Gotcha:
- Overengineering with abstract classes when not needed.
- Violating SRP while trying to achieve OCP.

#### 🎯 Interview Questions:
- When is OCP overkill?
    - Answer: If a feature is unlikely to change or extend, introducing interfaces and factories might add unnecessary complexity. For example, if there’s only one type of loyalty point calculator and it’s unlikely to change, overengineering OCP adds no value.

- Can you give a case where you extended functionality without touching core logic?
    - Answer: In a food delivery app, we had a PaymentProcessor interface. Adding Razorpay support didn’t require touching existing Stripe or PayPal logic — just implemented a new class and plugged it into the processor factory.

---

### 🔹 **3. Liskov Substitution Principle (LSP)**

> **"Objects of a superclass should be replaceable with objects of a subclass without breaking the app."**

#### ✅ Example: **Chess - Piece Movement**
- `Piece` superclass
  - `move(board, start, end)`
- Subclasses: `Bishop`, `Rook`, `Knight`, etc.

You should be able to do:
```python
def move_piece(piece: Piece, from_pos, to_pos):
    piece.move(from_pos, to_pos)
```
Whether it's a `Knight` or `Queen`, it shouldn't throw unexpected exceptions or break logic.

#### ❌ Violation:
- Having a subclass that overrides a method in a way that contradicts the base contract.

E.g. if `Pawn` overrides `move()` but throws "InvalidOperation" for backward move, it’s not substitutable.

#### ✅ Fix:
Use composition or rethink hierarchy (maybe `Pawn` has extra rules wrapped with validation logic instead of override).

#### 🔁 Common Gotcha:
- Adding restrictive logic in subclasses that violates the base class contract.

#### 🎯 Interview Questions:
- Can you give a real-world example of LSP violation?
    - Answer: In an airline booking system, we had a User base class and GuestUser subclass. GuestUser threw exceptions for some methods like getLoyaltyPoints() which were fine for RegisteredUser. That broke LSP since the subclass couldn’t be used safely as a base class.

- How would you refactor a system where subclasses break LSP?
    - Answer: Use composition instead of inheritance. In the above case, instead of forcing GuestUser to override and disable features, we moved loyalty logic into a separate interface and let only RegisteredUser use it.

---

### 🔹 **4. Interface Segregation Principle (ISP)**

> **"Clients should not be forced to depend on interfaces they don’t use."**

#### ✅ Example: **Restaurant - User Roles**
Suppose you have:
- `Admin`
- `Chef`
- `Waiter`

And an interface like:
```python
interface Staff {
    def take_order()
    def cook()
    def manage_inventory()
}
```

➡️ Now a `Waiter` must implement `cook()` 🤦‍♀️

#### ✅ Fix:
Split interfaces:
```python
interface OrderTaker { def take_order() }
interface Cook { def cook() }
interface InventoryManager { def manage_inventory() }
```

#### 🔁 Common Gotcha:
- Making giant interfaces like `IUser`, `IService`, etc. that get reused incorrectly.

#### 🎯 Interview Questions:
- Why not just have one big interface?
    - Answer: If multiple classes only use parts of a large interface, they are forced to implement methods they don’t need. This makes the system brittle. E.g., in a car rental system, mechanics don’t need access to payment processing methods — separate interfaces make responsibilities cleaner.

- Have you ever split an interface in a real system?
    - Answer: Yes, in a restaurant billing app, we had a single UserInterface with takeOrder(), generateBill(), and manageInventory(). We split it into OrderHandler, BillingHandler, and InventoryManager interfaces based on user roles.

---

### 🔹 **5. Dependency Inversion Principle (DIP)**

> **"Depend on abstractions, not concretions."**

#### ✅ Example: **Movie Ticket Booking - Payment**
```python
class BookingService:
    def __init__(self, paymentProcessor: IPaymentProcessor):
        self.payment = paymentProcessor
```

Now:
- Inject `PayPalProcessor`
- or `StripeProcessor`
- or `DummyPayment` for testing

#### ❌ Violation:
```python
class BookingService:
    def pay_with_paypal(self): ...
```
➡️ Too tightly coupled to PayPal.

#### ✅ Fix:
Use interfaces or abstract classes to **decouple** logic from specific implementations.

#### 🔁 Common Gotcha:
- Injecting concrete classes directly in constructors.
- Not using interfaces or mocking for testing.

#### 🎯 Interview Questions:
- Can you give a use case where you decoupled components with DIP?
    - Answer: In a stock trading app, the order executor originally depended directly on the broker API class. We refactored it to depend on an IBrokerAdapter interface. This allowed us to support multiple brokers and simplified testing.

- How does DIP help in unit testing?
    - Answer: By depending on abstractions, you can inject mock implementations during tests. For instance, a NotificationService that depends on IMailer lets you plug in a MockMailer for testing instead of hitting an actual SMTP server.

---

## 🚨 Final Interview Tips on SOLID

### 🔸 Patterns Interviewers Love:
- Talk about **breaking up monoliths**, **refactoring God classes**, or **injecting interfaces** to make things more testable.
- Use **real-world analogies** from web apps, banking, booking systems, etc.

### 🔸 Watch Out For:
- Overapplying SOLID = Overengineering
- “Manager” classes doing too much
- Mixing SRP and OCP — not every feature needs an interface

### 🔸 Reflective Questions to Prepare:
- How did you handle a change that broke a working system?
- How would you apply SOLID if you were designing a payment service?
- Do you think SOLID should always be applied, even under time pressure?



## ❓ Should you apply **all SOLID principles** while designing a system?

### ✅ **Short Answer:**
**No, not all at once.**  
And **definitely not prematurely.**

---

## 💡 **Long Answer:**

### 🎯 **SOLID is a guide, not a checklist.**
Think of SOLID principles like **tools** in your toolbox — you don’t hammer screws, right?

👉 Apply them **when a specific problem arises** or to **prevent a foreseeable one**.

---

### 🔄 **Real-World Design Is All About Trade-Offs**

Here’s how it usually plays out:

| Principle | When You Apply It | When You Might Delay It |
|----------|------------------|-------------------------|
| **SRP** (Single Responsibility) | Almost always useful early. Keeps classes clean and testable. | Might delay splitting unless complexity grows. |
| **OCP** (Open/Closed) | When you expect future extension (new payment methods, UI skins, etc.) | Don’t introduce abstract factories if there’s no current need. YAGNI (You Aren’t Gonna Need It). |
| **LSP** (Liskov Substitution) | Always validate when using inheritance. Useful in polymorphic systems. | If you’re not using inheritance, LSP may not even come into play. |
| **ISP** (Interface Segregation) | Makes sense in role-based systems (admin vs. user vs. guest). | If only 1-2 classes use an interface, keeping it unified is fine. |
| **DIP** (Dependency Inversion) | Always beneficial for **testability** and **decoupling**. | But overusing interfaces in very small systems can be overkill. |

---

## 🧠 Practical Advice from Real-World Engineers

- **Start simple** → Refactor using SOLID as complexity increases.
- Don’t **over-engineer** just to follow SOLID.
- Use principles to **solve** actual problems: tight coupling, hard-to-test code, etc.
- In early-stage systems or MVPs, prioritize **working code over perfect design**.

---

## 🚨 When Applying All SOLID Principles Becomes a Problem

1. **Premature abstraction** = maintenance nightmare.
2. **Too many interfaces** = "interface hell" with no real benefit.
3. **Misplaced inheritance** = leads to fragile base class problem.
4. **Decoupling everything** = can make a system harder to follow than necessary.

---

## ✅ Good Design Approach

Instead of asking:

> “Am I following all SOLID principles?”

Ask:

> “Is my design **easy to extend**, **test**, **understand**, and **maintain**?”

If the answer is yes — you're **already applying SOLID in spirit**, even if not explicitly ticking off each one.

---

### 💬 Interview Insight
If asked this in an interview, say:

> “I believe in using SOLID principles as **guiding heuristics**, not as rigid rules. I usually start with SRP and DIP early for testability, and bring in others like OCP and ISP as the system evolves and needs extension points.”

That shows **maturity** as a designer.


# Mapping Table

## 🔁 **Reverse Mapping Table**

### ✅ **Design Patterns**

| Pattern | Description | Case Studies |
|--------|-------------|--------------|
| **Singleton** | Ensure a class has only one instance (global access) | Parking Lot, Library System, Movie Booking (Theatre), Amazon (Inventory) |
| **Factory** | Creates objects without specifying exact class | ATM (Transaction), Car Rental, Hotel, Blackjack, Amazon (Product), Stack Overflow (Post) |
| **Strategy** | Select algorithm/behavior at runtime | Parking Lot (slot allocation), Payment Systems (Amazon, Movie Booking), Game Moves (Chess, Blackjack), Search (Library, Stack Overflow) |
| **Observer** | Notify multiple objects of state changes | Amazon (Order Status), Stack Overflow (Comments/Upvotes), Facebook/LinkedIn (Feeds), Cricinfo (Live Score), Airline (Flight status) |
| **Builder** | Step-by-step object creation | Cart (Amazon), Bill (Restaurant), Profile (LinkedIn/Facebook), Ticket (Airline) |
| **State** | Object behavior based on internal state | ATM, Chess, Blackjack, Movie Booking |
| **Command** | Encapsulate requests as objects (for undo/redo, etc.) | Stack Overflow (Moderation), Facebook (Like/Share), LinkedIn (Messaging) |
| **Decorator/Adapter** | Add behavior without modifying existing class | Less common in these case studies, optional |

---

### ✅ **SOLID Principles**

| Principle | Description | Case Studies |
|----------|-------------|--------------|
| **S**ingle Responsibility | One class = one responsibility | ALL (Every modular system needs SRP) |
| **O**pen/Closed | Open for extension, closed for modification | Amazon, Facebook, Library, ATM, Stack Overflow |
| **L**iskov Substitution | Subclass should be substitutable for parent | Chess (Piece types), ATM (Transaction types), Blackjack |
| **I**nterface Segregation | Prefer many small interfaces | Amazon (Search/Payment), Airline (Booking/Search), Facebook (Privacy) |
| **D**ependency Inversion | Depend on abstractions, not concretions | Parking Lot, Airline, ATM, Amazon |

---

### ✅ **OOP Concepts**

| Concept | Description | Case Studies |
|--------|-------------|--------------|
| **Inheritance** | IS-A relationship | Chess (Piece), ATM (Transaction), Card Games, Airline (User Types) |
| **Composition** | HAS-A relationship | Amazon (Cart/Product), Hotel (Room/Amenities), Restaurant (Menu/Order) |
| **Abstraction** | Hide implementation details | ATM, Airline, Library, Stock System |
| **Polymorphism** | Override behavior dynamically | Game Systems, Payment, Transaction, Search |
| **Encapsulation** | Protect internal state | ALL (standard best practice) |

---

## 🎯 **How to Use This Table**
Pick a **concept/design pattern** → Revise it with 1–2 **case studies** to implement.

### Example:
- Want to learn **Strategy Pattern**?  
  → Practice: *Parking Lot* (slot allocation), *Amazon* (Payment), *Chess* (Moves)

- Want to practice **Liskov Substitution Principle**?  
  → Try: *ATM* (Transactions), *Chess* (Piece behavior), *Blackjack*

- Want to get solid with **Observer Pattern**?  
  → Try: *Stack Overflow*, *Facebook*, *Cricinfo*, *Amazon*

